#  Self-Hosted Qwen2.5-VL Transcription

This notebook runs Qwen2.5-VL-3B-Instruct on a free Colab GPU to transcribe your handwritten exam answer images.

**Steps:**
1. Run the setup cell (installs dependencies — takes a couple of minutes)
2. Upload your handwritten answer images (Q1_variantA.jpg, etc.) using the upload cell
3. Run the transcription cell
4. Download `transcriptions.json` at the end

In [1]:
!pip install -q transformers accelerate qwen-vl-utils torch torchvision

import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None - go to Runtime > Change runtime type and select GPU')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 65.9 MB/s eta 0:00:00
GPU available: True
GPU name: Tesla T4


## Load the model
This downloads ~6GB of weights the first time 

In [2]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# min_pixels/max_pixels caps how many vision tokens a large image produces.
# Without this cap, a high-res phone photo can generate thousands of vision
# tokens and blow up GPU memory during attention -- this is the fix for the
# CUDA OutOfMemoryError. 1024*28*28 (~803K pixels, roughly 1024x784) is a
# resolution cap that comfortably preserves handwriting legibility while
# keeping memory use safe on a T4.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28

# float16, not bfloat16 -- T4 (Turing architecture) lacks native bfloat16
# tensor core support; bfloat16 either falls back to slow emulation or can
# contribute to memory issues. float16 is the correct choice on T4.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print('Model loaded.')

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded.


##  Transcription function
Same transcription prompt design as the API version — faithful transcription, equations as plain text, diagrams described in words.

In [8]:
TRANSCRIPTION_PROMPT = """You are transcribing a student's handwritten exam answer for automated grading.

Transcribe EXACTLY what is written in the image, including:
- All handwritten text, word for word
- Mathematical equations and formulas (write them in plain text, e.g. \"F = ma\", \"a = 4 m/s^2\")
- If there is a free-body diagram or sketch, describe it in words: what shapes/objects are drawn,
  what force vectors are shown (labeled with direction, e.g. \"arrow pointing up labeled N\"), and
  any labels present

Do not correct spelling or grammar. Do not add commentary or grade the answer. Just transcribe
faithfully what is on the page, in the same order it appears.

Output only the transcription, nothing else."""


def transcribe_image(image_path):
    # Free any cached memory from previous runs before allocating for this one
    torch.cuda.empty_cache()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": TRANSCRIPTION_PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    # inference_mode disables gradient tracking, reducing memory use during generation
    with torch.inference_mode():
      generated_ids = model.generate(
          **inputs,
          max_new_tokens=1000,
          repetition_penalty=1.2,
          no_repeat_ngram_size=3,
      )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    # Free memory used by this request before returning
    del inputs, generated_ids, generated_ids_trimmed
    torch.cuda.empty_cache()

    return output_text[0].strip()


print('Transcription function ready.')

Transcription function ready.


## Upload your handwritten answer images

In [4]:
from google.colab import files
import os

os.makedirs('student_answers', exist_ok=True)
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, os.path.join('student_answers', filename))

image_files = sorted([f for f in os.listdir('student_answers') if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'Uploaded {len(image_files)} images:', image_files)

Saving Q1_variantA.jpeg to Q1_variantA.jpeg
Saving Q1_variantB.jpeg to Q1_variantB.jpeg
Saving Q1_variantC.jpeg to Q1_variantC.jpeg
Saving Q2_variantA.jpeg to Q2_variantA.jpeg
Saving Q2_variantB.jpeg to Q2_variantB.jpeg
Saving Q2_variantC.jpeg to Q2_variantC.jpeg
Saving Q3_variantA.jpeg to Q3_variantA.jpeg
Saving Q3_variantB.jpeg to Q3_variantB.jpeg
Saving Q3_variantC.jpeg to Q3_variantC.jpeg
Saving Q4_variantA.jpeg to Q4_variantA.jpeg
Saving Q4_variantB.jpeg to Q4_variantB.jpeg
Saving Q4_variantC.jpeg to Q4_variantC.jpeg
Uploaded 12 images: ['Q1_variantA.jpeg', 'Q1_variantB.jpeg', 'Q1_variantC.jpeg', 'Q2_variantA.jpeg', 'Q2_variantB.jpeg', 'Q2_variantC.jpeg', 'Q3_variantA.jpeg', 'Q3_variantB.jpeg', 'Q3_variantC.jpeg', 'Q4_variantA.jpeg', 'Q4_variantB.jpeg', 'Q4_variantC.jpeg']


## Test on one image 

In [5]:
test_image = os.path.join('student_answers', image_files[0])
print(f'Testing on: {test_image}\n')

result = transcribe_image(test_image)
print('--- TRANSCRIPTION ---')
print(result)

Testing on: student_answers/Q1_variantA.jpeg

--- TRANSCRIPTION ---
Q1:
Ams: Newton's first law states that an object at rest stays at rest, and an object in motion continues moving at constant velocity, unless acted upon by a net external force. This is also called the law of inertia.
An inertial frame of reference in which Newton's 1st law holds true—that is, a frame that is either at rest or moving at constant velocity (not accelerating). In such a frame, an object with no net force acting on it will not appear to accelerate.
Eg: A ball placed on a table in a car moving at constant velocity on a straight road will stay still relative to the car, since the car is approximately an inertial frame. But if the car suddenly brakes, the ball will appear to roll forward—this is because the car's frame is no longer inertial (its accelerating).


## run the full batch

In [6]:
import json
import re

FILENAME_PATTERN = re.compile(r"(Q\d+)_variant([A-Za-z])", re.IGNORECASE)

results = []
for i, filename in enumerate(image_files, 1):
    image_path = os.path.join('student_answers', filename)
    match = FILENAME_PATTERN.search(filename)
    question_id = match.group(1).upper() if match else 'UNKNOWN'
    variant = match.group(2).upper() if match else 'UNKNOWN'

    print(f'[{i}/{len(image_files)}] Transcribing {filename}...')
    try:
        transcription = transcribe_image(image_path)
        status = 'success'
    except Exception as e:
        transcription = None
        status = f'error: {e}'
        print(f'  FAILED: {e}')

    results.append({
        'filename': filename,
        'question_id': question_id,
        'variant': variant,
        'transcription': transcription,
        'status': status,
    })

with open('transcriptions.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

succeeded = sum(1 for r in results if r['status'] == 'success')
print(f'\nDone. {succeeded}/{len(results)} transcribed successfully.')

[1/12] Transcribing Q1_variantA.jpeg...
[2/12] Transcribing Q1_variantB.jpeg...
[3/12] Transcribing Q1_variantC.jpeg...
[4/12] Transcribing Q2_variantA.jpeg...
[5/12] Transcribing Q2_variantB.jpeg...
[6/12] Transcribing Q2_variantC.jpeg...
[7/12] Transcribing Q3_variantA.jpeg...
[8/12] Transcribing Q3_variantB.jpeg...
[9/12] Transcribing Q3_variantC.jpeg...
[10/12] Transcribing Q4_variantA.jpeg...
[11/12] Transcribing Q4_variantB.jpeg...
[12/12] Transcribing Q4_variantC.jpeg...

Done. 12/12 transcribed successfully.


In [9]:
result = transcribe_image('student_answers/Q2_variantB.jpeg')
print(result)

2.
ẑ_1 z̃ _3 k g ^2 / n c → F = ma  
∴ 20 = 5a   
∴ a = 10m/s²


## Download transcriptions.json

In [7]:
from google.colab import files
files.download('transcriptions.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>